In [2]:
import pandas as pd

In [6]:
local = pd.read_csv('./data/topic-results/local_topic.csv')[['title', 'new_link', 'text', 'label']]
non_local = pd.read_csv('./data/topic-results/non_local_topic.csv')[['title', 'new_link', 'text', 'label']]

In [7]:
local.sample(frac=0.1, random_state=1024).to_csv('./data/validation-set/local.csv', index=None)

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    classification_report
)

### Local

In [35]:
file_path = './data/validation-set/macau_local_val.xlsx'
df = pd.read_excel(file_path)

df["true_label"] = df["true_label"].fillna("").astype(str).str.strip()
df["label"] = df["label"].fillna("").astype(str).str.strip()

df["final_true_label"] = np.where(
    df["true_label"] == "",
    df["label"],
    df["true_label"]
)

# 将字符串标签拆成集合 "政治与政策@经济与金融" -> ["政治与政策", "经济与金融"]
def split_labels(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [i.strip() for i in str(x).split("@") if i.strip()]

df["pred_list"] = df["label"].apply(split_labels)
df["true_list"] = df["final_true_label"].apply(split_labels)

all_labels = sorted(set(sum(df["pred_list"].tolist(), []) + sum(df["true_list"].tolist(), [])))
mlb = MultiLabelBinarizer(classes=all_labels)
y_pred = mlb.fit_transform(df["pred_list"])
y_true = mlb.transform(df["true_list"])

In [36]:
# 计算整体指标
exact_match_ratio = accuracy_score(y_true, y_pred)   # 子集准确率，要求整条样本标签完全一致
hloss = hamming_loss(y_true, y_pred)

micro_precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_true, y_pred, average="micro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

samples_precision = precision_score(y_true, y_pred, average="samples", zero_division=0)
samples_recall = recall_score(y_true, y_pred, average="samples", zero_division=0)
samples_f1 = f1_score(y_true, y_pred, average="samples", zero_division=0)

jaccard_micro = jaccard_score(y_true, y_pred, average="micro", zero_division=0)
jaccard_macro = jaccard_score(y_true, y_pred, average="macro", zero_division=0)
jaccard_samples = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

print("\n===== 整体评估指标 =====")
print(f"Exact Match Ratio : {exact_match_ratio:.4f}")
print(f"Hamming Loss      : {hloss:.4f}")

print(f"Micro Precision   : {micro_precision:.4f}")
print(f"Micro Recall      : {micro_recall:.4f}")
print(f"Micro F1          : {micro_f1:.4f}")

print(f"Macro Precision   : {macro_precision:.4f}")
print(f"Macro Recall      : {macro_recall:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")

print(f"Samples Precision : {samples_precision:.4f}")
print(f"Samples Recall    : {samples_recall:.4f}")
print(f"Samples F1        : {samples_f1:.4f}")

print(f"Jaccard Micro     : {jaccard_micro:.4f}")
print(f"Jaccard Macro     : {jaccard_macro:.4f}")
print(f"Jaccard Samples   : {jaccard_samples:.4f}")


===== 整体评估指标 =====
Exact Match Ratio : 0.9179
Hamming Loss      : 0.0082
Micro Precision   : 0.9451
Micro Recall      : 0.9909
Micro F1          : 0.9675
Macro Precision   : 0.9314
Macro Recall      : 0.9848
Macro F1          : 0.9554
Samples Precision : 0.9622
Samples Recall    : 0.9903
Samples F1        : 0.9712
Jaccard Micro     : 0.9370
Jaccard Macro     : 0.9169
Jaccard Samples   : 0.9593
